In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
#read in the places, shapefile and racialsed reviews
places_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/places_filtered.parquet'
shapefile_file = '/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp'
racialised_reviews_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/racialised_reclassed_reviews.parquet'
all_reviews_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/social_classified_text_reviews.parquet'
demographics_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/constituency_strukturdaten_28112025.csv'


In [ ]:
places_df = pd.read_parquet(places_file)
shapefile_gdf = gpd.read_file(shapefile_file)
racialised_reviews_df = pd.read_parquet(racialised_reviews_file)
all_reviews_df = pd.read_parquet(all_reviews_file)
demographics_df = pd.read_csv(demographics_file)

In [ ]:
demographics_df.columns = demographics_df.columns.str.strip() #strip whitespace from column names
demographics_df.columns

In [ ]:
#rename columns for clarity
shapefile_gdf = shapefile_gdf.rename(columns={'WKR_NR': 'constituency_number', 'WKR_NAME': 'constituency_name', 
                                              'LAND_NR': 'state_number', 'LAND_NAME': 'state_name'})


In [ ]:
#select columns from places and racialised reviews to merge on
places_cols = ['unique_identifier', 'place_id', 'place_name',
       'address', 'owner', 'borough', 'place_rating',
       'primary_place_type', 'place_subtype', 'latitude', 'longitude']

racialised_reviews_cols = ['review_id', 'unique_identifier', 'reviewer_id',
                           'org_review','org_review_lang', 'trans_review', 'reviewer_rating', 
                           'other_reviews','reviewer_name', 'review_location','retrieved_time',
                            'other_ratings']

demographic_cols = ['state_name', 'constituency_number', 'constituency_name',
                  'total_population', 'german_population', 'foreign_population']


In [ ]:
##create the new total_text_reviews column from  the all text reviews column in the all_reviews_df
#all_reviews_df['total_text_reviews'] = 
all_reviews_df['total_text_reviews'] = (
    all_reviews_df
    .groupby('unique_identifier')['review_id']
    .transform('count')
)

In [ ]:
racialised_reviews_df.head()

In [ ]:
#merge places and racialised reviews on unique identifier
merged_df = pd.merge(racialised_reviews_df[racialised_reviews_cols],
                      places_df[places_cols], on='unique_identifier', how='inner')
merged_df =pd.merge(merged_df,all_reviews_df[['review_id', 'total_text_reviews']], on='review_id', how='inner')
#drop all null columns
merged_df = merged_df.dropna(axis=1, how='all')
merged_df.info()

In [ ]:
### this was adding undesired points


# # Step 1: Reproject
# shapefile_gdf = shapefile_gdf.to_crs(epsg=3857)
# merged_gdf = gpd.GeoDataFrame(
#     merged_df,
#     geometry=gpd.points_from_xy(merged_df.longitude, merged_df.latitude),
#     crs='EPSG:4326'
# ).to_crs(epsg=3857)

# print("=== DIAGNOSTICS ===")
# print(f"merged_gdf points: {len(merged_gdf)}")
# print(f"shapefile polygons: {len(shapefile_gdf)}")
# print(f"shapefile CRS: {shapefile_gdf.crs}")
# print(f"merged_gdf CRS: {merged_gdf.crs}")
# print(f"shapefile geometry types: {shapefile_gdf.geometry.geom_type.value_counts().to_dict()}")
# print(f"any null geometries in points: {merged_gdf.geometry.isna().sum()}")
# print(f"any null geometries in shapefile: {shapefile_gdf.geometry.isna().sum()}")
# print(f"any invalid geometries in shapefile: {(~shapefile_gdf.geometry.is_valid).sum()}")

# # Check bounding boxes overlap
# print(f"\nPoints bounds:    {merged_gdf.total_bounds}")
# print(f"Shapefile bounds: {shapefile_gdf.total_bounds}")

# # Step 2: Dissolved filter
# dissolved = shapefile_gdf.geometry.union_all()
# shapefile_for_filter = gpd.GeoDataFrame(geometry=[dissolved], crs=shapefile_gdf.crs)
# shapefile_for_filter['geometry'] = shapefile_for_filter['geometry'].buffer(550000)

# print(f"\nDissolved geometry type: {shapefile_for_filter.geometry.geom_type[0]}")
# print(f"Dissolved geometry valid: {shapefile_for_filter.geometry.is_valid[0]}")

# # Step 3: Filter
# points_in_boundary = gpd.sjoin(
#     merged_gdf, shapefile_for_filter, how='inner', predicate='within'
# ).drop(columns=['index_right'])
# print(f"\nAfter dissolved filter: {len(points_in_boundary)}")

# # Step 4: nearest join
# spatial_join_gdf = gpd.sjoin_nearest(points_in_boundary, shapefile_gdf, how='left')
# print(f"After sjoin_nearest: {len(spatial_join_gdf)}")

# spatial_join_gdf = spatial_join_gdf.drop_duplicates(subset='review_id')
# spatial_join_gdf = spatial_join_gdf.drop(columns=['index_right'])
# print(f"After dedup: {len(spatial_join_gdf)}")

In [ ]:

# Step 1: Reproject
shapefile_gdf = shapefile_gdf.to_crs(epsg=3857)
merged_gdf = gpd.GeoDataFrame(
    merged_df,
    geometry=gpd.points_from_xy(merged_df.longitude, merged_df.latitude),
    crs='EPSG:4326'
).to_crs(epsg=3857)

# Step 2: Dissolve into one MultiPolygon — NO buffer
# This preserves all islands as separate polygons within the MultiPolygon
dissolved = shapefile_gdf.geometry.union_all()
germany_exact = gpd.GeoDataFrame(geometry=[dissolved], crs=shapefile_gdf.crs)

# Step 3: Strict containment — point must be inside actual Germany geometry
points_in_germany = gpd.sjoin(
    merged_gdf, germany_exact, how='inner', predicate='within'
).drop(columns=['index_right'])

print(f"Points strictly inside Germany (incl. islands): {len(points_in_germany)}")

# Step 4: Nearest join to assign attributes from the shapefile
spatial_join_gdf = gpd.sjoin_nearest(points_in_germany, shapefile_gdf, how='left')
spatial_join_gdf = spatial_join_gdf.drop_duplicates(subset='review_id')
spatial_join_gdf = spatial_join_gdf.drop(columns=['index_right'])

print(f"Final deduplicated count: {len(spatial_join_gdf)}")

In [ ]:
spatial_join_gdf.info()
#drop right index column



In [ ]:
#add demographic and political data
racialised_reviews_augmented = spatial_join_gdf.merge(demographics_df, on= ['constituency_number', 'constituency_name', 'state_name'], how='left')
racialised_reviews_augmented.info()

In [ ]:
#add the east/west variable
def classify_east_west(row):
    if row['state_name'] in ['Brandenburg', 'Mecklenburg-Vorpommern', 'Sachsen', 'Sachsen-Anhalt', 'Thüringen', 'Berlin']:
        return 'east'
    else: 
        return 'west'

racialised_reviews_augmented['east_west'] = racialised_reviews_augmented.apply(classify_east_west, axis=1)

## Prevalence of Racialised Discourse

In [ ]:
# Assuming racial_reviews is your main dataset
print(f"Total reviews: {len(racialised_reviews_augmented)}")
print(f"Total pools (unique place_id): {racialised_reviews_augmented['place_id'].nunique()}")

print(f"Number of states with at least one racialised connotation review: {len(racialised_reviews_augmented.state_name.unique())}")
print(f"Number of constituencies with at least one racialised connotation review: {len(racialised_reviews_augmented.constituency_name.unique())}")
print(f'Number of east reviews: {len(racialised_reviews_augmented[racialised_reviews_augmented["east_west"] == "east"])}')
print(f'Number of west reviews: {len(racialised_reviews_augmented[racialised_reviews_augmented["east_west"] == "west"])}')

In [ ]:
pool_totals = (racialised_reviews_augmented
               .drop_duplicates(subset='unique_identifier')
               .groupby('state_name')['total_text_reviews']
               .sum()
               .reset_index(name='total_reviews_state'))

print(pool_totals)
print(f"\nTotal reviews across all states: {pool_totals['total_reviews_state'].sum():,.0f}")
print(f"Total racialised reviews: {racialised_reviews_augmented['review_id'].nunique():,.0f}")
print(f"Overall pct: {racialised_reviews_augmented['review_id'].nunique() / pool_totals['total_reviews_state'].sum() * 100:.3f}%")

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

# --- Aggregate to state level from the main dataframe ---
# one row per review, so drop duplicates on constituency before summing population
constituency_pop = (racialised_reviews_augmented
                    .drop_duplicates(subset=['constituency_name', 'state_name'])
                    [['state_name', 'east_west', 'foreign_population', 'total_population']])

foreign_by_state = (constituency_pop
                    .groupby(['state_name', 'east_west'])
                    .agg(
                        foreign_population=('foreign_population', 'sum'),
                        total_population=('total_population', 'sum')
                    )
                    .reset_index())

foreign_by_state['foreign_pct'] = (foreign_by_state['foreign_population'] /
                                    foreign_by_state['total_population'] * 100).round(2)

# --- Racialised reviews as % of all reviews per state ---
# total_reviews column is the pool-level review count — take max per pool to avoid summing duplicates
pool_totals = (racialised_reviews_augmented
               .drop_duplicates(subset='unique_identifier')
               .groupby('state_name')['total_text_reviews']
               .sum()
               .reset_index(name='total_reviews_state'))

racialised_per_state = (racialised_reviews_augmented
                        .groupby('state_name')['review_id']
                        .nunique()
                        .reset_index(name='racialised_reviews'))

state_scatter = (foreign_by_state
                 .merge(pool_totals, on='state_name', how='left')
                 .merge(racialised_per_state, on='state_name', how='left'))

state_scatter['racialised_pct'] = (state_scatter['racialised_reviews'] /
                                    state_scatter['total_reviews_state'] * 100).round(2)

print(state_scatter[['state_name', 'east_west', 'foreign_pct', 
                      'racialised_reviews', 'total_reviews_state', 'racialised_pct']])

In [ ]:
# --- Scatter plot ---
ew_colors = {'west': 'steelblue', 'east': 'tomato'}

fig, ax = plt.subplots(figsize=(11, 7))

for ew, group in state_scatter.groupby('east_west'):
    color = ew_colors.get(ew, 'gray')
    ax.scatter(group['foreign_pct'], group['racialised_pct'],
               color=color, s=80, zorder=3, label=ew)
    for row in group.itertuples():
        ax.annotate(row.state_name,
                    (row.foreign_pct, row.racialised_pct),
                    textcoords='offset points', xytext=(6, 3),
                    fontsize=8, color=color)

# trend line
slope, intercept, r, p, _ = stats.linregress(
    state_scatter['foreign_pct'], state_scatter['racialised_pct']
)
x_line = np.linspace(state_scatter['foreign_pct'].min(),
                     state_scatter['foreign_pct'].max(), 100)
ax.plot(x_line, slope * x_line + intercept,
        color='gray', linestyle='--', linewidth=1,
        label=f'Trend (r={r:.2f}, p={p:.3f})')

ax.set_xlabel('Foreign population as % of state population')
ax.set_ylabel('Racialised reviews as % of all reviews in state')
ax.set_title('Foreign population share vs proportion of racialised reviews by state',
             fontweight='bold')
ax.legend(title='Region', fontsize=9)
ax.text(0.99, 0.01, f'n={len(state_scatter)} states',
        transform=ax.transAxes, fontsize=7, color='gray', ha='right', va='bottom')

plt.tight_layout()
plt.savefig('./visualisations/fig_foreign_pop_vs_racialised.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nr={r:.3f}, p={p:.3f}')
print(f'Slope: per 1pp increase in foreign population share, '
      f'racialised review proportion {"increases" if slope > 0 else "decreases"} by {abs(slope):.2f}pp')

In [ ]:
#distribution of reacialised reviews across pools, counstittuencites and states
reviews_per_pool = racialised_reviews_augmented.groupby('place_id').size().reset_index(name='review_count')
reviews_per_constituency = racialised_reviews_augmented.groupby('constituency_name').size().reset_index(name='review_count')
reviews_per_state = racialised_reviews_augmented.groupby('state_name').size().reset_index(name='review_count')  


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Prep data ---
pool_constituency = racialised_reviews_augmented.groupby(
    ['place_id', 'constituency_name', 'state_name']
).size().reset_index(name='review_count')

constituency_summary = pool_constituency.groupby(
    ['constituency_name', 'state_name']
).agg(
    pool_count=('place_id', 'count'),
    total_reviews=('review_count', 'sum')
).reset_index()

reviews_per_state = racialised_reviews_augmented.groupby('state_name').size().reset_index(name='review_count')
state_meta = racialised_reviews_augmented[['state_name', 'east_west']].drop_duplicates('state_name')
constituency_counts = constituency_summary.groupby('state_name')['constituency_name'].nunique().reset_index(name='constituency_count')

reviews_per_state = reviews_per_state.merge(state_meta, on='state_name').merge(constituency_counts, on='state_name')

# % of total reviews across all states
total_reviews = reviews_per_state['review_count'].sum()
reviews_per_state['review_pct'] = (reviews_per_state['review_count'] / total_reviews * 100).round(1)
reviews_per_state = reviews_per_state.sort_values('review_pct', ascending=True)

states = sorted(reviews_per_state['state_name'].unique())
ew_colors = {'west': 'steelblue', 'east': 'tomato'}

# ================================================================
# Figure 1: % of total reviews per state
# ================================================================
fig1, ax = plt.subplots(figsize=(12, 6))

bar_colors = reviews_per_state['east_west'].map(ew_colors).fillna('gray')
bars = ax.barh(reviews_per_state['state_name'], reviews_per_state['review_pct'], color=bar_colors)

max_val = reviews_per_state['review_pct'].max()
for bar, row in zip(bars, reviews_per_state.itertuples()):
    ax.text(row.review_pct + max_val * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{row.review_pct}%', va='center', ha='left', fontsize=9)
    ax.text(row.review_pct + max_val * 0.07, bar.get_y() + bar.get_height() / 2,
            f'({row.constituency_count} constituencies)',
            va='center', ha='left', fontsize=8, color='gray')

handles = [plt.Rectangle((0, 0), 1, 1, color=c, label=label) for label, c in ew_colors.items()]
ax.legend(handles=handles, title='Region', fontsize=9, title_fontsize=9)
ax.set_xlim(0, max_val * 1.35)
ax.set_xlabel('% of total racialised reviews')
ax.set_title('Total racialised reviews by state (% of all reviews)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('./visualisations/fig1_reviews_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# Figure 2+: One plot per state — % of state's reviews per constituency
# ================================================================
for state in states:
    state_data = constituency_summary[
        constituency_summary['state_name'] == state
    ].copy()

    # % within this state
    state_total = state_data['total_reviews'].sum()
    state_data['review_pct'] = (state_data['total_reviews'] / state_total * 100).round(1)
    state_data = state_data.sort_values('review_pct', ascending=True)

    ew = reviews_per_state.loc[reviews_per_state['state_name'] == state, 'east_west'].values[0]
    state_color = ew_colors.get(ew, 'gray')

    n = len(state_data)
    fig_height = max(4, n * 0.45)
    fig, ax = plt.subplots(figsize=(12, fig_height))

    bars = ax.barh(state_data['constituency_name'], state_data['review_pct'],
                   color=state_color, height=0.6)

    max_rev = state_data['review_pct'].max()
    for bar, row in zip(bars, state_data.itertuples()):
        ax.text(row.review_pct + max_rev * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{row.review_pct}%',
                va='center', ha='left', fontsize=8)
        ax.text(row.review_pct + max_rev * 0.08,
                bar.get_y() + bar.get_height() / 2,
                f'({row.pool_count} pool{"s" if row.pool_count != 1 else ""})',
                va='center', ha='left', fontsize=8, color='gray')

    median = state_data['review_pct'].median()
    ax.axvline(median, color='black', linestyle='--', linewidth=0.8,
               label=f'Median: {median:.1f}%')

    ax.set_xlim(0, max_rev * 1.3)
    ax.set_xlabel('% of state\'s racialised reviews')
    ax.set_title(f'{state} — share of reviews per constituency  [{ew}]',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f'./visualisations/fig_{state.lower().replace(" ", "_")}_constituencies.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## other dimensions to explore: themes,connotation,approach


In [ ]:
# load the new classification
approach_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_approach.parquet'
themes_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_theme.parquet'
connotation_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_connotation.parquet'
context_file = '/home/ethel/Desktop/swimming_pool_project/spp_15052026/data/reclassed_racialised_context.parquet'

#read all the files merge and explode them
theme_df = pd.read_parquet(themes_file)
context_df = pd.read_parquet(context_file)
connotation_df = pd.read_parquet(connotation_file)
approach_df = pd.read_parquet(approach_file)


racialised_dimensions = theme_df[['review_id', 'theme']].merge(context_df[['review_id', 'context']], on='review_id', how='outer')\
    .merge(connotation_df[['review_id', 'connotation']], on='review_id', how='outer')\
    .merge(approach_df[['review_id', 'approach']], on='review_id', how='outer')
    # drop null rows
racialised_dimensions = racialised_dimensions.dropna(subset=['theme', 'context', 'connotation', 'approach'], how='any').reset_index(drop=True) 

In [ ]:
# This is not valid JSON (no commas) so we parse it manually
def parse_list(val):
    if pd.isna(val):
        return []
    # strip brackets, split on whitespace, clean quotes
    val = str(val).strip().strip("[]")
    items = [v.strip().strip("'\"") for v in val.split("'") if v.strip().strip("'\"") != ',']
    #make lowercase
    items = [i.lower() for i in items]
    return [i for i in items if i]  # drop empty strings


In [ ]:
#parse and explode
for col in ['theme', 'context', 'connotation', 'approach']:
    racialised_dimensions[col] = racialised_dimensions[col].apply(parse_list)
    racialised_dimensions = racialised_dimensions.explode(col, ignore_index=True)

## Themes

In [ ]:
racialised_dimensions.theme.unique()

In [ ]:
racialised_dimensions_augmented = racialised_dimensions[['review_id', 'theme', 'context', 'connotation', 'approach']].merge(
    racialised_reviews_augmented[['review_id', 'state_name', 'constituency_name', 'east_west']], on='review_id', how='left')
# drop null rows


In [ ]:
racialised_dimensions_augmented.info()

In [ ]:
# Check how many unique combinations per review
racialised_dimensions_augmented.groupby("review_id").size().value_counts().sort_index()

In [ ]:
# Check per dimension how many reviews have multiple values
for col in ["theme", "connotation", "approach", "context"]:
    print(f"\n{col}:")
    print(racialised_dimensions_augmented.groupby("review_id")[col].nunique().value_counts().sort_index())

In [ ]:
for col in ['theme', 'context', 'connotation', 'approach']:
    racialised_dimensions[col] = racialised_dimensions[col].apply(parse_list)
    racialised_dimensions = racialised_dimensions.explode(col, ignore_index=True)

# Keep only reviews with multiple themes
# After exploding, theme is scalar, so count unique themes per review_id.
theme_counts_per_review = racialised_dimensions_augmented.groupby('review_id')['theme'].nunique()
multi_theme_review_ids = theme_counts_per_review[theme_counts_per_review > 1].index
multi_theme = racialised_dimensions_augmented[
    racialised_dimensions_augmented['review_id'].isin(multi_theme_review_ids)
]

In [ ]:

print(f"Rows after explode + merge: {len(racialised_dimensions_augmented[['theme']])}")
print(f"Nulls in east_west after merge: {racialised_dimensions_augmented[['east_west']].isna().sum()}")
print(racialised_dimensions_augmented['theme'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ew_colors = {'west': 'steelblue', 'east': 'tomato'}
KNOWN_THEMES = ['access', 'belonging', 'clothing', 'crowd', 'etiquette',
                'hygiene', 'language', 'other', 'safety', 'security', 'staff']

# ================================================================
# Figure 1: Overall distribution of reviews by theme
# ================================================================
# % of total reviews (a review can contribute to multiple themes, so % won't sum to 100)
total_reviews = racialised_dimensions_augmented['review_id'].nunique()
theme_counts = racialised_dimensions_augmented['theme'].value_counts().reindex(KNOWN_THEMES).sort_values()
theme_pct = (theme_counts / total_reviews * 100).round(1).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(theme_pct.index, theme_pct.values, color='steelblue')

max_val = theme_pct.max()
for bar, v in zip(bars, theme_pct.values):
    ax.text(v + max_val * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{v}%', va='center', fontsize=9)

ax.set_xlim(0, max_val * 1.15)
ax.set_xlabel('% of racialised reviews mentioning theme')
ax.set_title('Overall distribution of racialised reviews by theme', fontweight='bold')

# Note to clarify % basis
ax.text(0.99, 0.01, f'% of {total_reviews:,} reviews (multi-theme reviews counted once per theme)',
        transform=ax.transAxes, fontsize=7, color='gray', ha='right', va='bottom')

plt.tight_layout()
plt.savefig('./visualisations/fig_theme_overall.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# Figure 2: Themes by east/west — normalised to % for comparability
# ================================================================
ew_theme = racialised_dimensions_augmented.groupby(['east_west', 'theme']).size().unstack(fill_value=0)
ew_theme_pct = ew_theme.div(ew_theme.sum(axis=1), axis=0) * 100
ew_theme_pct = ew_theme_pct[KNOWN_THEMES]  # consistent column order

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(KNOWN_THEMES))
width = 0.35
for i, (ew, row) in enumerate(ew_theme_pct.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=ew,
                  color=ew_colors.get(ew, 'gray'))
    for bar, v in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{v:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(KNOWN_THEMES, rotation=35, ha='right')
ax.set_ylabel('% of reviews in region')
ax.set_title('Theme prevalence: east vs west (% of reviews per region)', fontweight='bold')
ax.legend(title='Region')
plt.tight_layout()
plt.savefig('./visualisations/fig_theme_east_west.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# Figure 3: Themes by state — normalised heatmap
# ================================================================
state_theme = racialised_dimensions_augmented.groupby(['state_name', 'theme']).size().unstack(fill_value=0)
state_theme_pct = state_theme.div(state_theme.sum(axis=1), axis=0) * 100
state_theme_pct = state_theme_pct[KNOWN_THEMES]

fig, ax = plt.subplots(figsize=(14, len(state_theme_pct) * 0.6 + 2))
im = ax.imshow(state_theme_pct.values, aspect='auto', cmap='Blues')

ax.set_xticks(range(len(KNOWN_THEMES)))
ax.set_xticklabels(KNOWN_THEMES, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(state_theme_pct)))
ax.set_yticklabels(state_theme_pct.index, fontsize=9)

# Annotate cells
for i in range(len(state_theme_pct)):
    for j in range(len(KNOWN_THEMES)):
        val = state_theme_pct.values[i, j]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                fontsize=8, color='white' if val > state_theme_pct.values.max() * 0.6 else 'black')

plt.colorbar(im, ax=ax, label='% of reviews in state')
ax.set_title('Theme prevalence by state (% of reviews per state)', fontweight='bold')
plt.tight_layout()
plt.savefig('./visualisations/fig_theme_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# Figure 4: Co-occurrence heatmap — reviews with 2+ themes
# ================================================================
from itertools import combinations

# Keep only reviews with multiple themes
#multi_theme = racialised_dimensions_augmented[racialised_dimensions_augmented['theme'].apply(len) > 1]
theme_counts_per_review = racialised_dimensions_augmented.groupby('review_id')['theme'].nunique()
multi_theme_review_ids = theme_counts_per_review[theme_counts_per_review > 1].index
multi_theme = racialised_dimensions_augmented[
    racialised_dimensions_augmented['review_id'].isin(multi_theme_review_ids)
]
print(multi_theme[['review_id', 'theme']].head())
# Build co-occurrence matrix
cooc = pd.DataFrame(0, index=KNOWN_THEMES, columns=KNOWN_THEMES)
for themes in multi_theme['theme']:
    for t1, t2 in combinations(sorted(themes), 2):
        if t1 in KNOWN_THEMES and t2 in KNOWN_THEMES:
            cooc.loc[t1, t2] += 1
            cooc.loc[t2, t1] += 1

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cooc.values, aspect='auto', cmap='YlOrRd')

ax.set_xticks(range(len(KNOWN_THEMES)))
ax.set_xticklabels(KNOWN_THEMES, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(KNOWN_THEMES)))
ax.set_yticklabels(KNOWN_THEMES, fontsize=9)

for i in range(len(KNOWN_THEMES)):
    for j in range(len(KNOWN_THEMES)):
        val = cooc.values[i, j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=7, color='white' if val > cooc.values.max() * 0.6 else 'black')

plt.colorbar(im, ax=ax, label='Number of reviews')
ax.set_title('Theme co-occurrence — reviews with 2+ racialised themes', fontweight='bold')
plt.tight_layout()
plt.savefig('./visualisations/fig_theme_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nReviews with single theme: {racialised_dimensions_augmented['theme'].apply(len).eq(1).sum()}")
print(f"Reviews with 2+ themes:    {racialised_dimensions_augmented['theme'].apply(len).gt(1).sum()}")
print(f"\nTop co-occurring pairs:")
pairs = [(t1, t2, cooc.loc[t1, t2]) for t1, t2 in combinations(KNOWN_THEMES, 2)]
pairs_df = pd.DataFrame(pairs, columns=['theme_1', 'theme_2', 'count']).sort_values('count', ascending=False)
print(pairs_df.head(10).to_string(index=False))

## Approach

In [ ]:
# load the new classification
nature_df = pd.read_csv(approach_file)  # adjust path
print(nature_df.head(10))
print(f"\nColumns: {nature_df.columns.tolist()}")
print(f"\nUnique values in nature/approach column: {nature_df['approach'].value_counts()}")  # adjust column name

In [ ]:
# --- Merge approach + themes + geography ---
approach_df = nature_df[['review_id', 'approach']]

# merge themes (already exploded) with approach and geography
themes_approach = themes_merged.merge(approach_df, on='review_id', how='left')

print(f"Nulls in approach after merge: {themes_approach['approach'].isna().sum()}")
print(themes_approach.groupby(['east_west', 'approach']).size())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

KNOWN_THEMES = ['Access', 'Belonging', 'Clothing', 'Crowd', 'Etiquette',
                'Hygiene', 'Language', 'Other', 'Safety', 'Security', 'Staff']
APPROACHES = ['Implicit', 'Explicit', 'Both']
approach_colors = {'Implicit': 'steelblue', 'Explicit': 'tomato', 'Both': 'goldenrod'}
regions = sorted(themes_approach['east_west'].dropna().unique())

# denominator: unique reviews per region (not exploded rows)
region_totals = (themes_approach.drop_duplicates('review_id')
                 .groupby('east_west')['review_id'].nunique())


for region in regions:
    region_data = themes_approach[themes_approach['east_west'] == region]
    total = region_totals[region]

    counts = (region_data.groupby(['theme_exploded', 'approach'])['review_id']
              .nunique()
              .unstack(fill_value=0)
              .reindex(index=KNOWN_THEMES, columns=APPROACHES, fill_value=0))

    pct = (counts / total * 100).round(1)

    fig, ax = plt.subplots(figsize=(13, 7))

    x = np.arange(len(KNOWN_THEMES))
    width = 0.25
    for i, approach in enumerate(APPROACHES):
        bars = ax.bar(x + i * width, pct[approach], width,
                      label=approach, color=approach_colors[approach], alpha=0.85)
        for bar, v in zip(bars, pct[approach]):
            if v >= 1:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.2,
                        f'{v:.0f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x + width)
    ax.set_xticklabels(KNOWN_THEMES, rotation=35, ha='right', fontsize=10)
    ax.set_ylabel('% of reviews in region')
    ax.set_ylim(0, pct.values.max() * 1.2)
    ax.set_title(f'Racialisation approach by theme — {region.capitalize()}  (n={total:,} reviews)',
                 fontsize=12, fontweight='bold')
    ax.legend(title='Approach', fontsize=9, title_fontsize=9)
    ax.text(0.99, 0.99, f'% of {total:,} reviews — multi-theme reviews counted once per theme',
            transform=ax.transAxes, fontsize=7, color='gray', ha='right', va='top')

    plt.tight_layout()
    plt.savefig(f'fig_approach_theme_{region.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
racialised_reviews_augmented.info()

## Connotation

In [ ]:
connotation_df = pd.read_csv(connotation_file)  # adjust path
print(connotation_df.head(10))
print(f"\nColumns: {connotation_df.columns.tolist()}")
print(f"\nValue counts:\n{connotation_df['connotation'].value_counts()}")  # adjust column name

In [ ]:
# explode connotations
connotation_exploded = (connotation_df[['review_id', 'connotation']]
                        .explode('connotation')
                        .rename(columns={'connotation': 'connotation_exploded'}))
connotation_exploded = connotation_exploded[
    connotation_exploded['connotation_exploded'].notna() & 
    (connotation_exploded['connotation_exploded'] != '')
]

# merge with geography, themes and approach
connotation_merged = (connotation_exploded
                      .merge(racialised_reviews_augmented[['review_id', 'state_name', 'east_west']]
                             .drop_duplicates('review_id'), on='review_id', how='left')
                      .merge(approach_df, on='review_id', how='left')
                      .merge(themes_df[['review_id', 'theme_list']], on='review_id', how='left'))

# explode themes too so we can cross with connotation
connotation_themes = (connotation_merged
                      .explode('theme_list')
                      .rename(columns={'theme_list': 'theme'}))
connotation_themes = connotation_themes[
    connotation_themes['theme'].notna() & 
    (connotation_themes['theme'] != '')
]

print(f"Rows after explode: {len(connotation_merged)}")
print(f"Nulls in east_west: {connotation_merged['east_west'].isna().sum()}")
print(f"Nulls in approach: {connotation_merged['approach'].isna().sum()}")
print(connotation_merged['connotation_exploded'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

KNOWN_CONNOTATIONS = ['Negative', 'Neutral', 'Positive']
connotation_colors = {'Negative': 'tomato', 'Neutral': 'steelblue', 'Positive': 'mediumseagreen'}
APPROACHES = ['Implicit', 'Explicit', 'Both']
regions = sorted(connotation_merged['east_west'].dropna().unique())
total_reviews = connotation_merged['review_id'].nunique()

# ================================================================
# Figure 1: Connotation by east/west
# ================================================================
ew_counts = (connotation_merged
             .groupby(['east_west', 'connotation_exploded'])['review_id']
             .nunique()
             .unstack(fill_value=0)
             .reindex(columns=KNOWN_CONNOTATIONS, fill_value=0))

ew_pct = ew_counts.div(ew_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(regions))
width = 0.25
for i, conn in enumerate(KNOWN_CONNOTATIONS):
    bars = ax.bar(x + i * width, ew_pct[conn], width,
                  label=conn, color=connotation_colors[conn], alpha=0.85)
    for bar, v in zip(bars, ew_pct[conn]):
        if v >= 1:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.3,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(regions, fontsize=10)
ax.set_ylabel('% of reviews in region')
ax.set_title('Connotation of racialisation: east vs west', fontweight='bold')
ax.legend(title='Connotation', fontsize=9)
plt.tight_layout()
plt.savefig('fig_connotation_east_west.png', dpi=150, bbox_inches='tight')
plt.show()

# ================================================================
# Figure 2: Connotation × theme — one plot per region
# ================================================================
region_totals = (connotation_themes.drop_duplicates('review_id')
                 .groupby('east_west')['review_id'].nunique())

for region in regions:
    region_data = connotation_themes[connotation_themes['east_west'] == region]
    total = region_totals[region]

    counts = (region_data.groupby(['theme', 'connotation_exploded'])['review_id']
              .nunique()
              .unstack(fill_value=0)
              .reindex(index=KNOWN_THEMES, columns=KNOWN_CONNOTATIONS, fill_value=0))
    pct = (counts / total * 100).round(1)

    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(KNOWN_THEMES))
    width = 0.25
    for i, conn in enumerate(KNOWN_CONNOTATIONS):
        bars = ax.bar(x + i * width, pct[conn], width,
                      label=conn, color=connotation_colors[conn], alpha=0.85)
        for bar, v in zip(bars, pct[conn]):
            if v >= 1:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.2,
                        f'{v:.0f}', ha='center', va='bottom', fontsize=7)

    ax.set_xticks(x + width)
    ax.set_xticklabels(KNOWN_THEMES, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('% of reviews in region')
    ax.set_title(f'Connotation by theme — {region.capitalize()}  (n={total:,})', fontweight='bold')
    ax.legend(title='Connotation', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'fig_connotation_theme_{region.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

# ================================================================
# Figure 3: Connotation × approach — one plot per region
# ================================================================
for region in regions:
    region_data = connotation_merged[connotation_merged['east_west'] == region]
    total = region_totals[region]

    counts = (region_data.groupby(['approach', 'connotation_exploded'])['review_id']
              .nunique()
              .unstack(fill_value=0)
              .reindex(index=APPROACHES, columns=KNOWN_CONNOTATIONS, fill_value=0))
    pct = (counts / total * 100).round(1)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(APPROACHES))
    width = 0.25
    for i, conn in enumerate(KNOWN_CONNOTATIONS):
        bars = ax.bar(x + i * width, pct[conn], width,
                      label=conn, color=connotation_colors[conn], alpha=0.85)
        for bar, v in zip(bars, pct[conn]):
            if v >= 1:
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.3,
                        f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(APPROACHES, fontsize=10)
    ax.set_ylabel('% of reviews in region')
    ax.set_title(f'Connotation by approach — {region.capitalize()}  (n={total:,})', fontweight='bold')
    ax.legend(title='Connotation', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'fig_connotation_approach_{region.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

# ================================================================
# Figure 4: Top 3 states
# ================================================================
state_share = (racialised_per_state
               .assign(pct=lambda x: (x['racialised_reviews'] / 
                                      x['racialised_reviews'].sum() * 100).round(1))
               .sort_values('pct', ascending=False))

print(state_share)
cumsum = state_share['pct'].cumsum()
top_states = state_share.loc[cumsum <= 60, 'state_name'].tolist()
# make sure we have at least 3
if len(top_states) < 3:
    top_states = state_share['state_name'].iloc[:3].tolist()
print(f"\nTop states: {top_states}")

In [ ]:
racialised_reviews_augmented.info()

In [ ]:
 racialised_reviews_df.info()

## Context

In [ ]:
## merge dataframes
racialised_approach_dimension = racialised_reviews_augmented.merge(approach_df, on='review_id', how='left')